In [9]:
# Debugging code for ARMT model testing
# Copy-paste this into a Jupyter cell

import torch
import numpy as np
import datasets
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, Optional
import sys

# Add the path to import ARMT models
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")

from transformers import (
    AutoConfig, AutoTokenizer,
    AutoModelForCausalLM
)
from modeling_armt.huggingface import *

In [38]:
class AssociativeRecurrentWrapperv2(AssociativeRecurrentWrapper):
    def forward(self, 
                input_ids=None, 
                labels=None, 
                labels_mask=None, 
                inputs_embeds=None, 
                attention_mask=None, 
                output_attentions=None, 
                output_hidden_states=None,
                input_segmented=False,
                output_only_last_segment=False,
                use_previous_batch_state=torch.zeros(1),
                num_items_in_batch=None,  # Added to handle HF Trainer compatibility
                segments=None, # Compatibility with RecurrentWrapperNoSegmentationGenerate
                **kwargs  # Added to handle any other unexpected kwargs
                ):
        if segments is None:
            if input_segmented:
                n_segs = input_ids.shape[1] if not (input_ids is None) else inputs_embeds.shape[1]
                segmented = [dict(
                    input_ids=input_ids[:, i] if not (input_ids is None) else None, 
                    inputs_embeds=inputs_embeds[:, i] if not (inputs_embeds is None) else None, 
                    attention_mask=attention_mask[:, i],
                    labels=labels[:, i] if not (labels is None) else None, 
                    labels_mask=labels_mask[:, i] if not (labels_mask is None) else None, 
                ) for i in range(n_segs)]
                labels = torch.cat([labels[:, i] for i in range(n_segs)], dim=1)
                if labels_mask is not None:
                    labels_mask = torch.cat([labels_mask[:, i] for i in range(n_segs)], dim=1)
            else:
                segmented = self.segment(input_ids=input_ids, inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels, labels_mask=labels_mask)
        else:
            segmented = segments
        
        cell_outputs = []
        if not use_previous_batch_state.all() or self.last_state is None:
            self.memory_cell.zero_mem()
            state = None
        else: 
            self.memory_cell.detach_mem()
            state = self.last_state
        next_seg_kwargs = dict(state=state)
        for seg_num, segment in enumerate(segmented):
            if seg_num != len(segmented) - 1:
                next_seg_len = segmented[seg_num + 1]['input_ids'].size(-1)
            else:
                next_seg_len = None
            # Pass num_items_in_batch to segment processing
            segment_with_kwargs = dict(**segment, **next_seg_kwargs)
            if kwargs.get('num_items_in_batch') is not None:
                segment_with_kwargs['num_items_in_batch'] = kwargs['num_items_in_batch']
            cell_out, next_seg_kwargs = self.process_segment(segment_with_kwargs, next_seg_len=next_seg_len)
            if (not output_only_last_segment) or (seg_num == len(segmented) - 1):
                cell_outputs.append(cell_out)

        out = self.process_outputs(cell_outputs, labels=labels, 
                                   labels_mask=labels_mask,
                                   output_attentions=output_attentions, 
                                   output_hidden_states=output_hidden_states,
                                   num_items_in_batch=kwargs.get('num_items_in_batch'))
        
        if not self.training:
            self.memory_cell.zero_mem()
            self.last_state = None
        return out

In [39]:
class ARMTForCausalLMv2(ARMTForCausalLM):
    def __init__(self, config: ARMTConfig, **kwargs):
        super().__init__(config, **kwargs)
        from transformers import AutoConfig, AutoModelForCausalLM
        
        # Build base model either from name (pretrained weights) or from provided config
        base_model = None
        if getattr(config, 'base_model_name', None) is not None and getattr(config, 'base_model_config', None) is not None:
            raise ValueError("Exactly one of `base_model_name` or `base_model_config` must be provided in ARMTConfig.")
        bm_cfg = getattr(config, 'base_model_config', None)
        if bm_cfg is not None:
            # Prefer explicit config when provided
            if isinstance(bm_cfg, PretrainedConfig) and getattr(bm_cfg, 'model_type', None) != ARMTConfig.model_type:
                resolved_cfg = bm_cfg
            elif isinstance(bm_cfg, dict):
                if 'model_type' not in bm_cfg:
                    raise ValueError("`base_model_config` dict must include a 'model_type' key (e.g., 'gpt_neox', 'llama').")
                config_cls_or_instance = AutoConfig.for_model(bm_cfg['model_type'])
                # If an instance was returned, update it; if a class was returned, construct from dict
                if isinstance(config_cls_or_instance, PretrainedConfig):
                    resolved_cfg = config_cls_or_instance
                    for k, v in bm_cfg.items():
                        setattr(resolved_cfg, k, v)
                else:
                    resolved_cfg = config_cls_or_instance.from_dict(bm_cfg)
            elif isinstance(bm_cfg, str):
                # Treat as a name or path to load a config
                resolved_cfg = AutoConfig.from_pretrained(bm_cfg)
            else:
                raise TypeError("`base_model_config` must be a transformers.PretrainedConfig, dict, or str (name/path)")
            base_model = AutoModelForCausalLM.from_config(resolved_cfg)
        elif getattr(config, 'base_model_name', None):
            base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name)
        else:
            raise ValueError("ARMTForCausalLM requires either `base_model_config` or `base_model_name` in ARMTConfig.")

        self.armt_config = config
        
        # Create the associative memory cell
        memory_cell = AssociativeMemoryCell(
            base_model=base_model,
            num_mem_tokens=config.num_mem_tokens,
            d_mem=config.d_mem,
            layers_attr=config.layers_attr,
            wrap_pos=config.wrap_pos,
            correction=config.correction,
            n_heads=config.n_heads,
            use_denom=config.use_denom,
            gating=config.gating,
            freeze_mem=config.freeze_mem,
            act_on=config.act_on,
            max_hop=config.max_hop,
            act_type=config.act_type,
            # Optional extras
            constant_depth=config.get('constant_depth', False),
            act_format=config.get('act_format', 'linear'),
            noisy_halting=config.get('noisy_halting', False),
            attend_to_previous_input=config.attend_to_previous_input,
            use_sink=config.use_sink
        )
        
        # Create the associative recurrent wrapper
        self.armt = AssociativeRecurrentWrapperv2(
            memory_cell,
            segment_size=config.segment_size,
            segment_alignment=config.segment_alignment,
            sliding_window=config.sliding_window,
            attend_to_previous_input=config.attend_to_previous_input,
            act_on=config.act_on,
            time_penalty=config.time_penalty
        )
    
    def forward(self, labels=None, *args, **kwargs):
        return self.armt(labels=labels, *args, **kwargs)

In [40]:

# Set environment variables
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Parameters from the shell script
class ExperimentArgs:
    def __init__(self):
        # From shell script
        self.per_device_batch_size = 64
        self.gradient_accumulation_steps = 1
        self.total_batch_size = 64
        self.data_path = "N8-K2V2-V62_1M"  # First dataset from the loop
        self.tokenizer_path = "/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/tokenizers/kv_alphabet_62"
        self.learning_rate = 1e-04  # First LR from the loop
        self.n_layer = 4
        self.n_head = 4
        self.n_embd = 128
        self.base_model = "llama"
        self.n_mem_tokens = 8
        self.n_ctrl_tokens = 0
        self.use_mem_proj = False
        self.mem_proj_mode = "proj"
        self.max_steps = 200000
        self.eval_steps = 500
        self.logging_steps = 500
        self.warmup_steps = 10000
        self.early_stopping_patience = 500
        self.seed = 142

# Create args instance
args = ExperimentArgs()

print(f"Testing ARMT with parameters:")
print(f"  Base model: {args.base_model}")
print(f"  Layers: {args.n_layer}, Heads: {args.n_head}, Embedding: {args.n_embd}")
print(f"  Memory tokens: {args.n_mem_tokens}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Data path: {args.data_path}")

Testing ARMT with parameters:
  Base model: llama
  Layers: 4, Heads: 4, Embedding: 128
  Memory tokens: 8
  Learning rate: 0.0001
  Data path: N8-K2V2-V62_1M


In [41]:

# Create tokenizer
print("\n1. Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_path)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

# Create model config
print("\n2. Creating model config...")
if args.base_model == 'llama':
    config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
    config.num_hidden_layers = args.n_layer
    config.num_attention_heads = args.n_head
    config.num_key_value_heads = args.n_head
    config.hidden_size = args.n_embd
    config.head_dim = config.hidden_size // config.num_attention_heads
    config.intermediate_size = config.hidden_size * 4
else:
    raise ValueError(f'Unsupported base model: {args.base_model}')

config.torch_dtype = "float32"
config.vocab_size = tokenizer.vocab_size
config.pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')

print(f"Config vocab size: {config.vocab_size}")

# Create RMT config
print("\n3. Creating RMT config...")
rmt_config = ARMTConfig()
rmt_config.base_model_config = config
rmt_config.num_mem_tokens = args.n_mem_tokens
rmt_config.max_n_segments = 10
rmt_config.think_token_id = tokenizer.convert_tokens_to_ids('[THINK]')
rmt_config.answer_token_id = tokenizer.convert_tokens_to_ids('[ANSWER]')
rmt_config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
rmt_config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')

# Create model
print("\n4. Creating ARMT model...")
model = ARMTForCausalLMv2(rmt_config)
model.main_input_name = 'labels'
print(f"Model created: {type(model)}")


1. Loading tokenizer...
Tokenizer vocab size: 70

2. Creating model config...
Config vocab size: 70

3. Creating RMT config...

4. Creating ARMT model...
Model created: <class '__main__.ARMTForCausalLMv2'>


In [42]:

# Load dataset
print("\n5. Loading dataset...")
dataset = datasets.load_dataset(f"yurakuratov/{args.data_path}")
print(f"Dataset loaded: {dataset}")

# Define collate function (same as in the original script)
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    segments_batch = []
    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}


5. Loading dataset...
Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['context', 'query', 'target'],
        num_rows: 1000000
    })
    valid: Dataset({
        features: ['context', 'query', 'target'],
        num_rows: 5000
    })
})


In [43]:

# Test with a small batch
print("\n6. Testing with small batch...")
batch = [dataset['train'][i] for i in range(2)]  # Small batch for testing
print(f"Sample data:")
for i, sample in enumerate(batch):
    print(f"  Sample {i}:")
    print(f"    Context: {sample['context'][:100]}...")
    print(f"    Query: {sample['query']}")
    print(f"    Target: {sample['target']}")

# Collate the batch
collated = collate_fn(batch)
print(f"\nCollated batch shapes:")
print(f"  Segments: {len(collated['segments'])}")
print(f"  Segment 0 input_ids: {collated['segments'][0]['input_ids'].shape}")
print(f"  Segment 1 input_ids: {collated['segments'][1]['input_ids'].shape}")
print(f"  Labels: {collated['labels'].shape}")


6. Testing with small batch...
Sample data:
  Sample 0:
    Context: !V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|...
    Query: ?!I9:
    Target: dt!|
  Sample 1:
    Context: !wO:em!!nB:zb!!Mq:tS!!Bi:Nf!!Bp:Zp!!wM:nt!!MP:Sj!!5o:iR!|...
    Query: ?!wM:
    Target: nt!|

Collated batch shapes:
  Segments: 2
  Segment 0 input_ids: torch.Size([2, 57])
  Segment 1 input_ids: torch.Size([2, 9])
  Labels: torch.Size([2, 66])


In [44]:

# Move to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n7. Moving model and data to {device}...")
model = model.to(device)

# Move collated data to device
for k, v in collated.items():
    if isinstance(v, torch.Tensor):
        collated[k] = v.to(device)
for i, s in enumerate(collated['segments']):
    for k, v in s.items():
        if isinstance(v, torch.Tensor):
            collated['segments'][i][k] = v.to(device)

# Test forward pass
print("\n8. Testing forward pass...")
try:
    with torch.no_grad():
        outputs = model(**collated)
    
    print(f"Forward pass successful!")
    print(f"  Output keys: {list(outputs.keys())}")
    print(f"  Loss: {outputs.loss.item() if hasattr(outputs, 'loss') and outputs.loss is not None else 'N/A'}")
    print(f"  Logits shape: {outputs.logits.shape if hasattr(outputs, 'logits') else 'N/A'}")
    
    # Test token generation
    if hasattr(outputs, 'logits'):
        predicted_tokens = outputs.logits.argmax(dim=-1)
        print(f"\nPredicted tokens shape: {predicted_tokens.shape}")
        
        # Decode some predictions
        print(f"\nSample predictions:")
        for i in range(min(2, predicted_tokens.shape[0])):
            pred_text = tokenizer.decode(predicted_tokens[i], skip_special_tokens=True)
            print(f"  Sample {i}: {pred_text[:100]}...")
            
except Exception as e:
    print(f"Error during forward pass: {e}")
    import traceback
    traceback.print_exc()


7. Moving model and data to cuda...

8. Testing forward pass...
Forward pass successful!
  Output keys: ['loss', 'ce_loss', 'logits', 'logits_0', 'ce_loss_0', 'logits_1', 'ce_loss_1']
  Loss: 4.24569034576416
  Logits shape: torch.Size([2, 66, 70])

Predicted tokens shape: torch.Size([2, 66])

Sample predictions:
  Sample 0: G Q Q : Q p ! ! d k : j 4 ! ! Q j : B Q ! ! B B : B B ! ! c Q : B B ! ! y Q : B d ! ! w B : B o ! ! ...
  Sample 1: G w B : B B ! z z B : z b ! z M q : t z B B B i : N o G G B p : Z p G G w M : n t G G M P : z j ! G ...


In [45]:

print("\n9. Testing with different batch sizes...")
# Test with different batch sizes
for batch_size in [1, 4, 8]:
    try:
        print(f"\nTesting batch size {batch_size}...")
        batch = [dataset['train'][i] for i in range(batch_size)]
        collated = collate_fn(batch)
        
        # Move to device
        for k, v in collated.items():
            if isinstance(v, torch.Tensor):
                collated[k] = v.to(device)
        for i, s in enumerate(collated['segments']):
            for k, v in s.items():
                if isinstance(v, torch.Tensor):
                    collated['segments'][i][k] = v.to(device)
        
        with torch.no_grad():
            outputs = model(**collated)
        
        print(f"  Batch size {batch_size}: SUCCESS")
        print(f"    Loss: {outputs.loss.item() if hasattr(outputs, 'loss') and outputs.loss is not None else 'N/A'}")
        print(f"    Logits shape: {outputs.logits.shape if hasattr(outputs, 'logits') else 'N/A'}")
        
    except Exception as e:
        print(f"  Batch size {batch_size}: FAILED - {e}")

print("\n10. Testing memory usage...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size (MB): {sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024:.2f}")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB")
    print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024 / 1024:.2f} MB")

print("\n✅ Debugging complete! The model should be ready for training.")


9. Testing with different batch sizes...

Testing batch size 1...
  Batch size 1: SUCCESS
    Loss: 4.223489284515381
    Logits shape: torch.Size([1, 66, 70])

Testing batch size 4...
  Batch size 4: SUCCESS
    Loss: 4.325535774230957
    Logits shape: torch.Size([4, 66, 70])

Testing batch size 8...
  Batch size 8: SUCCESS
    Loss: 4.337155342102051
    Logits shape: torch.Size([8, 66, 70])

10. Testing memory usage...
Model parameters: 1,650,052
Model size (MB): 6.29
GPU memory allocated: 130.18 MB
GPU memory cached: 276.00 MB

✅ Debugging complete! The model should be ready for training.


In [13]:
# Additional debugging code for trainer testing
# Copy-paste this into a Jupyter cell after the previous debugging code

import accelerate
import transformers
from transformers import (
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback, 
    TrainerCallback,
    HfArgumentParser
)
from accelerate.logging import get_logger
import logging
import json
from pathlib import Path

# Set up logging
logger_fmt = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
log_lvl = logging.INFO
logging.basicConfig(format=logger_fmt, level=log_lvl)
logger = logging.getLogger('')

# Initialize accelerator
print("\n11. Setting up Accelerator...")
accel = accelerate.Accelerator()
logger = get_logger('')
transformers.utils.logging.set_verbosity(log_lvl)

logger.info(f'num processes: {accel.num_processes}')
logger.info(f'mixed precision: {accel.mixed_precision}')

# Define compute metrics function (same as in original script)
def compute_metrics_fn(eval_pred, ignore_token_ids, tokenizer):
    # Shift logits and labels for next-token prediction
    predictions, labels, inputs = eval_pred.predictions, eval_pred.label_ids, eval_pred.inputs
    
    logits = predictions
    print("[compute_metrics_fn] logits type:", type(logits))
    print("[compute_metrics_fn] logits len:", len(logits))
    if isinstance(logits, (list, tuple)):
        for i, l in enumerate(logits):
            print(f"[compute_metrics_fn] logits[{i}] type: {type(l)}, shape: {getattr(l, 'shape', None)}")
    else:
        print("[compute_metrics_fn] logits shape:", getattr(logits, 'shape', None))
    print("[compute_metrics_fn] labels type:", type(labels), "shape:", getattr(labels, 'shape', None))
    print("[compute_metrics_fn] inputs type:", type(inputs), "shape:", getattr(inputs, 'shape', None))
    
    logits = logits[..., :-1, :]
    labels = labels[..., 1:]
    preds = np.argmax(logits, axis=-1)

    # Create a mask for tokens that are not padding (-100) and ignored tokens (like ! and |)
    mask = (labels != -100)
    for t_id in ignore_token_ids:
        mask &= (labels != t_id)

    # Calculate token-level accuracy only on content tokens
    masked_predictions = preds[mask]
    masked_labels = labels[mask]

    accuracy = (masked_predictions == masked_labels).mean()

    # get exact_match per-sample accuracy, ignore masked tokens
    exact_match = np.mean([
        np.all(pred[mask[i]] == lab[mask[i]])
        for i, (pred, lab) in enumerate(zip(preds, labels))
        if np.any(mask[i])  # Skip samples that are all masked
    ])

    # Print some examples
    for pred, label, inp in zip(preds[:3], labels[:3], inputs[:3]):
        mask_sample = (label != -100)
        pred_sample = pred[mask_sample]
        inp_sample = inp.copy()
        label_sample = label.copy()
        inp_sample[inp_sample == -100] = 0
        label_sample[label_sample == -100] = 0
        print('i:', tokenizer.decode(inp_sample, skip_special_tokens=True).replace(' ', ''))
        print('p:', tokenizer.decode(pred_sample, skip_special_tokens=True).replace(' ', ''))
        print('t:', tokenizer.decode(label_sample, skip_special_tokens=True).replace(' ', ''))
        print('-' * 30)

    return {
        "token_accuracy": float(accuracy),
        "exact_match": float(exact_match),
    }

# Custom trainer class (same as in original script)
class CustomTrainer(Trainer):
    def create_scheduler(self, num_training_steps: int, optimizer: torch.optim.Optimizer = None):
        num_training_steps = int(num_training_steps / 0.9)  # to make final lr not zero, for linear it is lr/10.
        return super().create_scheduler(num_training_steps, optimizer)

    def log(self, logs: Dict[str, float], start_time: Optional[float] = None) -> None:
        # log early stopping patience
        for cb in self.callback_handler.callbacks:
            if isinstance(cb, EarlyStoppingCallback):
                logs['patience'] = cb.early_stopping_patience_counter
                break
        return super().log(logs, start_time=start_time)

# Stop on metric value callback
class StopOnMetricValue(TrainerCallback):
    def __init__(self, metric_name: str, value: float, higher_is_better: bool = True):
        self.metric_name = metric_name
        self.value = value
        self.higher_is_better = higher_is_better

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        if not self.metric_name.startswith("eval_"):
            metric_to_check = f"eval_{self.metric_name}"
        metric_value = metrics.get(metric_to_check)
        if metric_value is None:
            return
        operator = np.greater_equal if self.higher_is_better else np.less_equal
        if operator(metric_value, self.value):
            control.should_training_stop = True
            logger.info(f'metric {self.metric_name}={metric_value:.4f} >= {self.value:.4f}, stopping training..')

# Prepare dataset for training
print("\n12. Preparing datasets...")
train_dataset = dataset['train'].select(range(100))  # Small subset for testing
eval_dataset = dataset['valid'].select(range(20))   # Small subset for testing

print(f"Train dataset size: {len(train_dataset)}")
print(f"Eval dataset size: {len(eval_dataset)}")

2025-09-15 16:26:11,841 - root - INFO - num processes: 1
2025-09-15 16:26:11,842 - root - INFO - mixed precision: no



11. Setting up Accelerator...

12. Preparing datasets...
Train dataset size: 100
Eval dataset size: 20


In [14]:

# Target sequence looks like: "XXXX!|"
# Let's not count ! and | in the accuracy calculation
ignore_token_ids = [tokenizer.convert_tokens_to_ids(t) for t in ['!', '|']]

# Define custom compute metrics function with ignored tokens
def compute_metrics(eval_preds):
    return compute_metrics_fn(eval_preds, ignore_token_ids, tokenizer)

# Set up training arguments (reduced for testing)
print("\n13. Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="./debug_output",
    logging_dir="./debug_output",
    
    max_steps=10,  # Very few steps for testing
    per_device_train_batch_size=2,  # Small batch size
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    warmup_steps=2,
    weight_decay=0.0,
    learning_rate=args.learning_rate,
    lr_scheduler_type='constant_with_warmup',
    
    eval_strategy='steps',
    save_strategy='steps',
    save_steps=5,
    eval_steps=5,
    logging_steps=1,
    report_to=None,  # Disable wandb/tensorboard for debugging
    metric_for_best_model='token_accuracy',
    load_best_model_at_end=False,
    eval_on_start=True,
    greater_is_better=True,
    remove_unused_columns=False,
    include_num_input_tokens_seen=False,
    include_for_metrics=['inputs'],
    save_total_limit=1,
    dataloader_num_workers=0,  # Disable multiprocessing for debugging
    dataloader_pin_memory=False,
    seed=args.seed,
    fp16=False,  # Disable mixed precision for debugging
    bf16=False,
)

print(f"Training arguments configured:")
print(f"  Max steps: {training_args.max_steps}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Eval steps: {training_args.eval_steps}")

# Initialize Trainer
print("\n14. Initializing Trainer...")
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),
        StopOnMetricValue(metric_name='exact_match', value=1.0, higher_is_better=True),
    ],
)

print(f"Trainer initialized successfully!")

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).



13. Setting up training arguments...
Training arguments configured:
  Max steps: 10
  Batch size: 2
  Learning rate: 0.0001
  Eval steps: 5

14. Initializing Trainer...


max_steps is given, it will override any value given in num_train_epochs


Trainer initialized successfully!


In [16]:
initial_metrics = trainer.evaluate()



***** Running Evaluation *****
  Num examples = 20
  Batch size = 2


[compute_metrics_fn] logits type: <class 'tuple'>
[compute_metrics_fn] logits len: 6
[compute_metrics_fn] logits[0] type: <class 'numpy.ndarray'>, shape: (10,)
[compute_metrics_fn] logits[1] type: <class 'numpy.ndarray'>, shape: (20, 66, 70)
[compute_metrics_fn] logits[2] type: <class 'numpy.ndarray'>, shape: (20, 57, 70)
[compute_metrics_fn] logits[3] type: <class 'numpy.ndarray'>, shape: (10,)
[compute_metrics_fn] logits[4] type: <class 'numpy.ndarray'>, shape: (20, 9, 70)
[compute_metrics_fn] logits[5] type: <class 'numpy.ndarray'>, shape: (10,)
[compute_metrics_fn] labels type: <class 'numpy.ndarray'> shape: (20, 66)
[compute_metrics_fn] inputs type: <class 'numpy.ndarray'> shape: (20, 66)


TypeError: tuple indices must be integers or slices, not tuple

In [15]:

# Test evaluation before training
print("\n15. Running initial evaluation...")
try:
    initial_metrics = trainer.evaluate()
    print(f"Initial evaluation metrics: {initial_metrics}")
except Exception as e:
    print(f"Error during initial evaluation: {e}")
    import traceback
    traceback.print_exc()

# Test a few training steps
print("\n16. Running training steps...")
try:
    # Train for a few steps
    trainer.train()
    print("Training completed successfully!")
    
    # Final evaluation
    print("\n17. Running final evaluation...")
    final_metrics = trainer.evaluate()
    print(f"Final evaluation metrics: {final_metrics}")
    
except Exception as e:
    print(f"Error during training: {e}")
    import traceback
    traceback.print_exc()

# Test model saving and loading
print("\n18. Testing model saving...")
try:
    # Save the model
    save_path = "./debug_model_save"
    trainer.save_model(save_path)
    print(f"Model saved to {save_path}")
    
    # Test loading
    from modeling_armt.huggingface import ARMTForCausalLMv2, ARMTConfig
    loaded_model = ARMTForCausalLMv2.from_pretrained(save_path)
    print("Model loaded successfully!")
    
    # Test loaded model
    test_batch = [dataset['train'][0]]
    test_collated = collate_fn(test_batch)
    
    # Move to device
    for k, v in test_collated.items():
        if isinstance(v, torch.Tensor):
            test_collated[k] = v.to(device)
    for i, s in enumerate(test_collated['segments']):
        for k, v in s.items():
            if isinstance(v, torch.Tensor):
                test_collated['segments'][i][k] = v.to(device)
    
    with torch.no_grad():
        loaded_output = loaded_model(**test_collated)
    
    print(f"Loaded model test - Loss: {loaded_output.loss.item() if hasattr(loaded_output, 'loss') else 'N/A'}")
    print("Model saving/loading test successful!")
    
except Exception as e:
    print(f"Error during model saving/loading: {e}")
    import traceback
    traceback.print_exc()

# Test different learning rates
print("\n19. Testing different learning rates...")
learning_rates = [1e-5, 1e-4, 1e-3]
for lr in learning_rates:
    try:
        print(f"\nTesting learning rate: {lr}")
        
        # Create new model for each test
        test_model = ARMTForCausalLMv2(rmt_config)
        test_model.main_input_name = 'labels'
        test_model = test_model.to(device)
        
        # Create trainer with new learning rate
        test_args = TrainingArguments(
            output_dir=f"./debug_lr_{lr}",
            max_steps=3,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=1,
            learning_rate=lr,
            eval_strategy='no',
            save_strategy='no',
            logging_steps=1,
            report_to=None,
            remove_unused_columns=False,
            include_num_input_tokens_seen=False,
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            seed=args.seed,
        )
        
        test_trainer = CustomTrainer(
            model=test_model,
            args=test_args,
            train_dataset=train_dataset.select(range(5)),
            eval_dataset=None,
            data_collator=collate_fn,
            compute_metrics=None,
        )
        
        # Train for a few steps
        test_trainer.train()
        
        # Test forward pass
        test_batch = [dataset['train'][0]]
        test_collated = collate_fn(test_batch)
        for k, v in test_collated.items():
            if isinstance(v, torch.Tensor):
                test_collated[k] = v.to(device)
        for i, s in enumerate(test_collated['segments']):
            for k, v in s.items():
                if isinstance(v, torch.Tensor):
                    test_collated['segments'][i][k] = v.to(device)
        
        with torch.no_grad():
            test_output = test_model(**test_collated)
        
        print(f"  LR {lr}: SUCCESS - Loss: {test_output.loss.item():.4f}")
        
    except Exception as e:
        print(f"  LR {lr}: FAILED - {e}")

print("\n20. Memory and performance summary...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size (MB): {sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024:.2f}")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB")
    print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024 / 1024:.2f} MB")

print("\n✅ Trainer debugging complete! The model and training pipeline are ready.")


***** Running Evaluation *****
  Num examples = 20
  Batch size = 2



15. Running initial evaluation...


[compute_metrics_fn] logits type: <class 'tuple'>
[compute_metrics_fn] logits len: 6
[compute_metrics_fn] logits[0] type: <class 'numpy.ndarray'>, shape: (10,)
[compute_metrics_fn] logits[1] type: <class 'numpy.ndarray'>, shape: (20, 66, 70)
[compute_metrics_fn] logits[2] type: <class 'numpy.ndarray'>, shape: (20, 57, 70)
[compute_metrics_fn] logits[3] type: <class 'numpy.ndarray'>, shape: (10,)
[compute_metrics_fn] logits[4] type: <class 'numpy.ndarray'>, shape: (20, 9, 70)
[compute_metrics_fn] logits[5] type: <class 'numpy.ndarray'>, shape: (10,)
[compute_metrics_fn] labels type: <class 'numpy.ndarray'> shape: (20, 66)
[compute_metrics_fn] inputs type: <class 'numpy.ndarray'> shape: (20, 66)
Error during initial evaluation: tuple indices must be integers or slices, not tuple

16. Running training steps...


Traceback (most recent call last):
  File "/tmp/ipykernel_39659/2000920355.py", line 4, in <module>
    initial_metrics = trainer.evaluate()
                      ^^^^^^^^^^^^^^^^^^
  File "/workspace-SR006.nfs2/bulatov/envs/rmt/lib/python3.11/site-packages/transformers/trainer.py", line 4154, in evaluate
    output = eval_loop(
             ^^^^^^^^^^
  File "/workspace-SR006.nfs2/bulatov/envs/rmt/lib/python3.11/site-packages/transformers/trainer.py", line 4443, in evaluation_loop
    metrics = self.compute_metrics(
              ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_39659/3817090086.py", line 7, in compute_metrics
    return compute_metrics_fn(eval_preds, ignore_token_ids, tokenizer)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_39659/2531618186.py", line 49, in compute_metrics_fn
    logits = logits[..., :-1, :]
             ~~~~~~^^^^^^^^^^^^^
TypeError: tuple indices must be integers or slices, not tuple
***** Running training

Error during training: Error uploading run: returned error 403: {"data":null,"errors":[{"message":"\u003c!doctype html\u003e\u003cmeta charset=\"utf-8\"\u003e\u003cmeta name=viewport content=\"width=device-width, initial-scale=1\"\u003e\u003ctitle\u003e403\u003c/title\u003e403 Forbidden"}]}

18. Testing model saving...
Model saved to ./debug_model_save


...

  LR 1e-05: FAILED - Error uploading run: returned error 403: {"data":null,"errors":[{"message":"\u003c!doctype html\u003e\u003cmeta charset=\"utf-8\"\u003e\u003cmeta name=viewport content=\"width=device-width, initial-scale=1\"\u003e\u003ctitle\u003e403\u003c/title\u003e403 Forbidden"}]}

Testing learning rate: 0.0001


max_steps is given, it will override any value given in num_train_epochs
***** Running training *****
  Num examples = 5
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 1
  Gradient Accumulation steps = 1
  Total optimization steps = 3
  Number of trainable parameters = 1,650,052
Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"
Instantiating LlamaForCausalLM model under default dtype torch.float32.
Generate config GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 2,
  "pad_token_id": 0
}

Instantiating LlamaForCausalLM model under default dtype torch.float32.
Generate config GenerationConfig {
  "bos_token_id": 1,
  "eos_token_id": 2,
  "pad_token_id": 0
}

PyTorch: setting up devices


  LR 0.0001: FAILED - Error uploading run: returned error 403: {"data":null,"errors":[{"message":"\u003c!doctype html\u003e\u003cmeta charset=\"utf-8\"\u003e\u003cmeta name=viewport content=\"width=device-width, initial-scale=1\"\u003e\u003ctitle\u003e403\u003c/title\u003e403 Forbidden"}]}

Testing learning rate: 0.001


The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
max_steps is given, it will override any value given in num_train_epochs
***** Running training *****
  Num examples = 5
  Num Epochs = 1
  Instantaneous batch size per device = 1
  Total train batch size (w. parallel, distributed & accumulation) = 1
  Gradient Accumulation steps = 1
  Total optimization steps = 3
  Number of trainable parameters = 1,650,052
Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"


  LR 0.001: FAILED - Error uploading run: returned error 403: {"data":null,"errors":[{"message":"\u003c!doctype html\u003e\u003cmeta charset=\"utf-8\"\u003e\u003cmeta name=viewport content=\"width=device-width, initial-scale=1\"\u003e\u003ctitle\u003e403\u003c/title\u003e403 Forbidden"}]}

20. Memory and performance summary...
Model parameters: 1,650,052
Model size (MB): 6.29
GPU memory allocated: 44.69 MB
GPU memory cached: 48.00 MB

✅ Trainer debugging complete! The model and training pipeline are ready.


In [2]:
import torch
import datasets
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from torch.nn.utils.rnn import pad_sequence

/workspace-SR006.nfs2/bulatov/envs/rmt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ARMT

In [3]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_armt.huggingface import ARMTForCausalLM, ARMTConfig

*** Can't import RWKV model ***
[2025-09-15 16:18:53,889] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-15 16:18:56,601] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [ ]:
# Debugging code for ARMT model testing
# Copy-paste this into a Jupyter cell

import torch
import numpy as np
import datasets
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, Optional
import sys

# Add the path to import ARMT models
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")

from transformers import (
    AutoConfig, AutoTokenizer,
    AutoModelForCausalLM
)
from modeling_armt.huggingface import ARMTForCausalLMv2, ARMTConfig

# Set environment variables
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Parameters from the shell script
class ExperimentArgs:
    def __init__(self):
        # From shell script
        self.per_device_batch_size = 64
        self.gradient_accumulation_steps = 1
        self.total_batch_size = 64
        self.data_path = "N8-K2V2-V62_1M"  # First dataset from the loop
        self.tokenizer_path = "./tokenizers/kv_alphabet_62/"
        self.learning_rate = 1e-04  # First LR from the loop
        self.n_layer = 4
        self.n_head = 4
        self.n_embd = 128
        self.base_model = "llama"
        self.n_mem_tokens = 8
        self.n_ctrl_tokens = 0
        self.use_mem_proj = False
        self.mem_proj_mode = "proj"
        self.max_steps = 200000
        self.eval_steps = 500
        self.logging_steps = 500
        self.warmup_steps = 10000
        self.early_stopping_patience = 500
        self.seed = 142

# Create args instance
args = ExperimentArgs()

print(f"Testing ARMT with parameters:")
print(f"  Base model: {args.base_model}")
print(f"  Layers: {args.n_layer}, Heads: {args.n_head}, Embedding: {args.n_embd}")
print(f"  Memory tokens: {args.n_mem_tokens}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Data path: {args.data_path}")

# Create tokenizer
print("\n1. Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_path)
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

# Create model config
print("\n2. Creating model config...")
if args.base_model == 'llama':
    config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
    config.num_hidden_layers = args.n_layer
    config.num_attention_heads = args.n_head
    config.num_key_value_heads = args.n_head
    config.hidden_size = args.n_embd
    config.head_dim = config.hidden_size // config.num_attention_heads
    config.intermediate_size = config.hidden_size * 4
else:
    raise ValueError(f'Unsupported base model: {args.base_model}')

config.torch_dtype = "float32"
config.vocab_size = tokenizer.vocab_size
config.pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')

print(f"Config vocab size: {config.vocab_size}")

# Create RMT config
print("\n3. Creating RMT config...")
rmt_config = ARMTConfig()
rmt_config.base_model_config = config
rmt_config.num_mem_tokens = args.n_mem_tokens
rmt_config.max_n_segments = 10
rmt_config.think_token_id = tokenizer.convert_tokens_to_ids('[THINK]')
rmt_config.answer_token_id = tokenizer.convert_tokens_to_ids('[ANSWER]')
rmt_config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
rmt_config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')

# Create model
print("\n4. Creating ARMT model...")
model = ARMTForCausalLMv2(rmt_config)
model.main_input_name = 'labels'
print(f"Model created: {type(model)}")

# Load dataset
print("\n5. Loading dataset...")
dataset = datasets.load_dataset(f"yurakuratov/{args.data_path}")
print(f"Dataset loaded: {dataset}")

# Define collate function (same as in the original script)
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    segments_batch = []
    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

# Test with a small batch
print("\n6. Testing with small batch...")
batch = [dataset['train'][i] for i in range(2)]  # Small batch for testing
print(f"Sample data:")
for i, sample in enumerate(batch):
    print(f"  Sample {i}:")
    print(f"    Context: {sample['context'][:100]}...")
    print(f"    Query: {sample['query']}")
    print(f"    Target: {sample['target']}")

# Collate the batch
collated = collate_fn(batch)
print(f"\nCollated batch shapes:")
print(f"  Segments: {len(collated['segments'])}")
print(f"  Segment 0 input_ids: {collated['segments'][0]['input_ids'].shape}")
print(f"  Segment 1 input_ids: {collated['segments'][1]['input_ids'].shape}")
print(f"  Labels: {collated['labels'].shape}")

# Move to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\n7. Moving model and data to {device}...")
model = model.to(device)

# Move collated data to device
for k, v in collated.items():
    if isinstance(v, torch.Tensor):
        collated[k] = v.to(device)
for i, s in enumerate(collated['segments']):
    for k, v in s.items():
        if isinstance(v, torch.Tensor):
            collated['segments'][i][k] = v.to(device)

# Test forward pass
print("\n8. Testing forward pass...")
try:
    with torch.no_grad():
        outputs = model(**collated)
    
    print(f"Forward pass successful!")
    print(f"  Output keys: {list(outputs.keys())}")
    print(f"  Loss: {outputs.loss.item() if hasattr(outputs, 'loss') and outputs.loss is not None else 'N/A'}")
    print(f"  Logits shape: {outputs.logits.shape if hasattr(outputs, 'logits') else 'N/A'}")
    
    # Test token generation
    if hasattr(outputs, 'logits'):
        predicted_tokens = outputs.logits.argmax(dim=-1)
        print(f"\nPredicted tokens shape: {predicted_tokens.shape}")
        
        # Decode some predictions
        print(f"\nSample predictions:")
        for i in range(min(2, predicted_tokens.shape[0])):
            pred_text = tokenizer.decode(predicted_tokens[i], skip_special_tokens=True)
            print(f"  Sample {i}: {pred_text[:100]}...")
            
except Exception as e:
    print(f"Error during forward pass: {e}")
    import traceback
    traceback.print_exc()

print("\n9. Testing with different batch sizes...")
# Test with different batch sizes
for batch_size in [1, 4, 8]:
    try:
        print(f"\nTesting batch size {batch_size}...")
        batch = [dataset['train'][i] for i in range(batch_size)]
        collated = collate_fn(batch)
        
        # Move to device
        for k, v in collated.items():
            if isinstance(v, torch.Tensor):
                collated[k] = v.to(device)
        for i, s in enumerate(collated['segments']):
            for k, v in s.items():
                if isinstance(v, torch.Tensor):
                    collated['segments'][i][k] = v.to(device)
        
        with torch.no_grad():
            outputs = model(**collated)
        
        print(f"  Batch size {batch_size}: SUCCESS")
        print(f"    Loss: {outputs.loss.item() if hasattr(outputs, 'loss') and outputs.loss is not None else 'N/A'}")
        print(f"    Logits shape: {outputs.logits.shape if hasattr(outputs, 'logits') else 'N/A'}")
        
    except Exception as e:
        print(f"  Batch size {batch_size}: FAILED - {e}")

print("\n10. Testing memory usage...")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model size (MB): {sum(p.numel() * p.element_size() for p in model.parameters()) / 1024 / 1024:.2f}")

if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024 / 1024:.2f} MB")
    print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024 / 1024:.2f} MB")

print("\n✅ Debugging complete! The model should be ready for training.")

In [4]:
base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M")
# tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")
tokenizer = AutoTokenizer.from_pretrained("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/tokenizers/kv_alphabet_62")

In [6]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)['train']

In [31]:
context = dataset[0]['context']
query = dataset[0]['query']
target = dataset[0]['target']



In [1]:
class Holder:
    pass

args = Holder()
args.n_layer = 4
args.n_head = 4
args.n_embd = 128
args.memory_task_freq = 0.5
args.memory_task = "reconstruct"
args.memory_key_size = 4
args.memory_value_size = 4


In [ ]:
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    segments_batch = []
    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

In [24]:
batch = [dataset[i] for i in range(10)]
collated = collate_fn(batch)


In [ ]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_armt.huggingface import AssociativeMemoryCell, AssociativeRecurrentWrapper, ARMTForCausalLM, ARMTConfig, PretrainedConfig

class AssociativeRecurrentWrapperNoSegmentationGenerate(AssociativeRecurrentWrapper):
    def forward(self, segments, labels, output_attentions=None, output_hidden_states=None,
                output_only_last_segment=False,
                use_previous_batch_state=torch.zeros(1),
                *args, **kwargs):
        """
        segments: list of dicts, each with keys like 'input_ids', 'attention_mask', etc.
        labels: not used here, but kept for interface compatibility.
        output_attentions, output_hidden_states: optional flags.
        Returns: list of outputs, one per segment.
        """
        memory_state = None
        cell_outputs = []
        for seg_num, segment in enumerate(segments):
            # Prepare kwargs for memory_cell
            seg_kwargs = dict(segment)
            if memory_state is not None:
                seg_kwargs['state'] = memory_state
            seg_kwargs['output_hidden_states'] = True
            cell_out = self.memory_cell(**seg_kwargs)
            cell_outputs.append(cell_out)
            memory_state = cell_out.get('state', None)

        cell_outputs = []
        if not use_previous_batch_state.all() or self.last_state is None:
            self.memory_cell.zero_mem()
            state = None
        else: 
            self.memory_cell.detach_mem()
            state = self.last_state
        next_seg_kwargs = dict(state=state)
        for seg_num, segment in enumerate(segments):
            if seg_num != len(segments) - 1:
                next_seg_len = segments[seg_num + 1]['input_ids'].size(-1)
            else:
                next_seg_len = None
            # Pass num_items_in_batch to segment processing
            segment_with_kwargs = dict(**segment, **next_seg_kwargs)
            if kwargs.get('num_items_in_batch') is not None:
                segment_with_kwargs['num_items_in_batch'] = kwargs['num_items_in_batch']
            cell_out, next_seg_kwargs = self.process_segment(segment_with_kwargs, next_seg_len=next_seg_len)
            if (not output_only_last_segment) or (seg_num == len(segments) - 1):
                cell_outputs.append(cell_out)

        out = self.process_outputs(cell_outputs, labels=labels, 
                                   labels_mask=torch.cat([s['labels_mask'] for s in segments], dim=1),
                                   output_attentions=output_attentions, 
                                   output_hidden_states=output_hidden_states,
                                   num_items_in_batch=kwargs.get('num_items_in_batch'))
        
        if not self.training:
            self.memory_cell.zero_mem()
            self.last_state = None
        return out

class ARMTForCausalLMv2(ARMTForCausalLM):
    def __init__(self, config: ARMTConfig, **kwargs):
        super().__init__(config, **kwargs)
        from transformers import AutoConfig, AutoModelForCausalLM
        
        # Build base model either from name (pretrained weights) or from provided config
        base_model = None
        if getattr(config, 'base_model_name', None) is not None and getattr(config, 'base_model_config', None) is not None:
            raise ValueError("Exactly one of `base_model_name` or `base_model_config` must be provided in ARMTConfig.")
        bm_cfg = getattr(config, 'base_model_config', None)
        if bm_cfg is not None:
            # Prefer explicit config when provided
            if isinstance(bm_cfg, PretrainedConfig) and getattr(bm_cfg, 'model_type', None) != ARMTConfig.model_type:
                resolved_cfg = bm_cfg
            elif isinstance(bm_cfg, dict):
                if 'model_type' not in bm_cfg:
                    raise ValueError("`base_model_config` dict must include a 'model_type' key (e.g., 'gpt_neox', 'llama').")
                config_cls_or_instance = AutoConfig.for_model(bm_cfg['model_type'])
                # If an instance was returned, update it; if a class was returned, construct from dict
                if isinstance(config_cls_or_instance, PretrainedConfig):
                    resolved_cfg = config_cls_or_instance
                    for k, v in bm_cfg.items():
                        setattr(resolved_cfg, k, v)
                else:
                    resolved_cfg = config_cls_or_instance.from_dict(bm_cfg)
            elif isinstance(bm_cfg, str):
                # Treat as a name or path to load a config
                resolved_cfg = AutoConfig.from_pretrained(bm_cfg)
            else:
                raise TypeError("`base_model_config` must be a transformers.PretrainedConfig, dict, or str (name/path)")
            base_model = AutoModelForCausalLM.from_config(resolved_cfg)
        elif getattr(config, 'base_model_name', None):
            base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name)
        else:
            raise ValueError("ARMTForCausalLM requires either `base_model_config` or `base_model_name` in ARMTConfig.")

        self.armt_config = config
        
        # Create the associative memory cell
        memory_cell = AssociativeMemoryCell(
            base_model=base_model,
            num_mem_tokens=config.num_mem_tokens,
            d_mem=config.d_mem,
            layers_attr=config.layers_attr,
            wrap_pos=config.wrap_pos,
            correction=config.correction,
            n_heads=config.n_heads,
            use_denom=config.use_denom,
            gating=config.gating,
            freeze_mem=config.freeze_mem,
            act_on=config.act_on,
            max_hop=config.max_hop,
            act_type=config.act_type,
            # Optional extras
            constant_depth=config.get('constant_depth', False),
            act_format=config.get('act_format', 'linear'),
            noisy_halting=config.get('noisy_halting', False),
            attend_to_previous_input=config.attend_to_previous_input,
            use_sink=config.use_sink
        )
        # print(memory_cell)
        
        # Create the associative recurrent wrapper
        self.armt = AssociativeRecurrentWrapperNoSegmentationGenerate(
            memory_cell,
            segment_size=config.segment_size,
            segment_alignment=config.segment_alignment,
            sliding_window=config.sliding_window,
            attend_to_previous_input=config.attend_to_previous_input,
            act_on=config.act_on,
            time_penalty=config.time_penalty
        )
    
    def forward(self, *args, **kwargs):
        return self.armt(*args, **kwargs)


In [70]:
config = ARMTConfig(
                 base_model_name="HuggingFaceTB/SmolLM2-360M",
                 base_model_config=None,
                 num_mem_tokens=16,
                 d_mem=32,
                 segment_size=512,
                 segment_alignment="left",
                 sliding_window=False,
                 attend_to_previous_input=False,
                 use_sink=False,
                 layers_attr="model.layers",
                 wrap_pos=False,
                 correction=True,
                 n_heads=1,
                 use_denom=True,
                 gating=False,
                 freeze_mem=False,
                 act_on=False,
                 max_hop=4,
                 act_type="associative",
                 act_format="linear",
                 noisy_halting=False,
                 constant_depth=False,
                 time_penalty=0.0,)

In [71]:

armt = ARMTForCausalLMv2(config)
armt.to('cuda')
':)'

':)'

In [72]:
# to cuda
for k, v in collated.items():
    if isinstance(v, torch.Tensor):
        collated[k] = v.to('cuda')
':)'
for i, s in enumerate(collated['segments']):
    for k, v in s.items():
        if isinstance(v, torch.Tensor):
            collated['segments'][i][k] = v.to('cuda')
':)'

':)'

In [73]:
out = armt(**collated)

In [75]:
out.loss

tensor(7.2220, device='cuda:0', grad_fn=<DivBackward0>)

In [77]:
collated['labels'].shape

torch.Size([10, 114])

In [76]:
out.logits.shape

torch.Size([10, 114, 49152])

In [ ]:
tokenizer.batch_decode(out.logits.argmax(dim=-1).cpu().numpy())


### interpret collate

In [25]:
collated['segments'][0]['input_ids'].shape, collated['segments'][1]['input_ids'].shape

(torch.Size([10, 57]), torch.Size([10, 57]))

In [26]:
tokenizer.batch_decode(collated['segments'][0]['input_ids'])

['! V 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t ! |',
 '! w O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R ! |',
 '! a O : K x ! ! y A : 6 2 ! ! r O : i S ! ! W i : 1 l ! ! G J : n i ! ! p o : D D ! ! 4 3 : z k ! ! C 6 : 6 i ! |',
 '! S J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N ! |',
 '! q y : l x ! ! N b : r K ! ! 0 D : O a ! ! 7 f : V r ! ! z Z : x 7 ! ! z N : q 3 ! ! I L : n K ! ! 7 j : 3 Z ! |',
 '! q K : x l ! ! h E : l J ! ! P 9 : Q g ! ! o 6 : D G ! ! K W : 6 w ! ! L z : B W ! ! D j : x l ! ! g n : 4 o ! |',
 '! z V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r ! |',
 '! a a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R ! |',
 '! f m : Y k ! ! e J : X t ! ! 0 V : W I ! ! p d : q 3 

In [27]:
tokenizer.batch_decode(collated['segments'][1]['input_ids'])

['! ? ! ? 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t',
 '! ? ! ? O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R',
 '? ! a O : K x ! | [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]',
 '! ? ! ? J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N',
 '? ! 7 f : V r ! | [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]',
 '? ! P 9 : Q g ! | [PAD] 

In [28]:
for l, m in zip(collated['segments'][1]['input_ids'], collated['segments'][1]['labels_mask']):
    print(tokenizer.decode(l[m]))


? ! ? 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t
? ! ? O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R
: K x ! |
? ! ? J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N
: V r ! |
: Q g ! |
? ! ? V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r
? ! ? a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R
: W I ! |
? ! ? 4 : u r ! ! T U : f I ! ! W F : H x ! ! Z M : P B ! ! l W : B 5 ! ! L 8 : c E ! ! W G : 4 R ! ! w l : q p


In [ ]:
collated['segments']['']

[{'input_ids': tensor([[   17,    70,    40,    42, 20091, 10095, 30641,    42,    90,    36,
           10095,    65,    90,    42,    37,    64, 10095,    56,    56,    42,
             102,   101, 10095,    83,    66,    42,    93,    39, 10095,   105,
              64,    42,    39,    84, 10095,   103,    93,    42,    71,    58,
           10095,    57,    41,    42, 14149,    17,   108,     0,     0],
          [   17,   103,    63,    42,   391, 10095,    94,    50,    42,   106,
              82, 10095,    61,    97,    42,   100,    67, 10095, 19514,    42,
              62,    86, 10095,    50,    96,    42,    74,    96, 10095,   103,
              61,    42,   399, 10095,  7015,    42,    67,    90, 10095,    37,
              95,    42,    89,    66,    17,   108,     0,     0,     0],
          [   17,    81,    63,    42,    59,   104, 10095,   105,    49,    42,
              38,    34, 10095,    98,    63,    42,    89,    67, 10095,    71,
              89,    42,   

In [16]:
collated['input_ids'].shape

KeyError: 'input_ids'

In [10]:
armt.to('cuda')
# to cuda
for k, v in collated.items():
    if isinstance(v, torch.Tensor):
        collated[k] = v.to('cuda')
':)'

':)'

In [11]:
out = armt(**collated)

In [31]:
# print(armt)

In [12]:
out.logits.shape

torch.Size([10, 98, 50432])

In [13]:
tokenizer.batch_decode(out.logits.argmax(dim=-1).cpu().numpy())

['uilt dystrophyUIDmergedise discover cause spilled soldier recall discover closest iOSChoose Gut discover discover Defensepoints和½\x18 discover Updated leather和 aboard automated discover Fa discover和 automated           discoverermalode和 leather Liberal discover y          和 Golf          ergecodescodes Abs Abs southwest Kolk y hexagonal CU collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections',
 'uiltdocs visionmerge powered discover postmodern envymerge specimen splendid discover Oper�mergeUID enthusiasts discover marvelous cries Gut Mexican

In [7]:
# type(armt)(config)

In [14]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_armt.huggingface import ARMTForCausalLM, ARMTConfig


*** Can't import RWKV model ***


In [15]:
config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
config.num_hidden_layers = 4
config.num_attention_heads = 4
config.num_key_value_heads = 4
config.hidden_size = 128
config.head_dim = config.hidden_size // config.num_attention_heads
config.intermediate_size = config.hidden_size * 4

config.torch_dtype = "float32"  # weights in float32, at training precision is controlled by accelerate
# config.vocab_size = 70
config.pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')


rmt_config = ARMTConfig()
rmt_config.base_model_config = config
rmt_config.num_mem_tokens = 32
rmt_config.max_n_segments = 10
rmt_config.think_token_id = tokenizer.convert_tokens_to_ids('[THINK]')
rmt_config.answer_token_id = tokenizer.convert_tokens_to_ids('[ANSWER]')
rmt_config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
rmt_config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')

model = ARMTForCausalLM(rmt_config)
model.main_input_name = 'labels'


In [16]:
# model

In [21]:
model.to('cuda')
':)'

':)'

In [22]:
out = model(**collated)

In [23]:
collated['input_ids'].shape

torch.Size([10, 2, 49])

In [24]:
out.loss

tensor(8.6703, device='cuda:0', grad_fn=<DivBackward0>)

In [1]:
torch.ones(10, 10)

NameError: name 'torch' is not defined

In [26]:
out.keys()

odict_keys(['loss', 'ce_loss', 'logits', 'logits_0', 'ce_loss_0', 'logits_1', 'ce_loss_1'])

In [25]:
out.logits.shape

torch.Size([10, 98, 128256])

RMT

In [ ]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)

In [3]:
ds = dataset['train']

In [4]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/tokenizers/kv_alphabet_62")

In [6]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_rmt.huggingface import RMTForReasoning, RMTConfig


[2025-09-01 11:56:57,732] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-01 11:56:59,903] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [29]:
from transformers import AutoConfig, AutoModelForCausalLM
cfg_name = "HuggingFaceTB/SmolLM2-360M"
model_cfg = AutoConfig.from_pretrained(cfg_name)
base_model = AutoModelForCausalLM.from_config(model_cfg, use_flash_attn=True)


TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'use_flash_attn'

In [24]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (rotary_emb): LlamaRotaryEm

In [ ]:
class Holder:
    pass

args = Holder()
args.n_layer = 4
args.n_head = 4
args.n_embd = 128


In [8]:
from transformers import AutoConfig
from transformers import AutoModelForCausalLM

base_model_config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
base_model_config.num_hidden_layers = args.n_layer
base_model_config.num_attention_heads = args.n_head
base_model_config.num_key_value_heads = args.n_head
base_model_config.hidden_size = args.n_embd
base_model_config.head_dim = base_model_config.hidden_size // base_model_config.num_attention_heads
base_model_config.intermediate_size = base_model_config.hidden_size * 4

In [9]:

# base_model = AutoModelForCausalLM.from_config(config)

In [10]:
config = RMTConfig()
# config.base_model_name = "HuggingFaceTB/SmolLM2-135M"
config.base_model_config = base_model_config
config.num_mem_tokens = 16
config.max_n_segments = 10
config.think_token_id = 100
config.answer_token_id = 101
config.bos_token_id = 102
config.eos_token_id = 103

model = RMTForReasoning(config)

# model.load_state_dict(torch.load("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/models/N8-K2V2-V62_1M/model.pt"))

In [11]:
from modeling_rmt.language_modeling import MemoryCell, RecurrentWrapper

In [12]:
model.main_input_name

'input_ids'

In [13]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [14]:
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    segments_batch = []
    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

In [ ]:
batch = [ds[i] for i in range(10)]
collated = collate_fn(batch)


In [16]:
import torch

In [17]:
model.to(dtype=torch.bfloat16)
out = model(**collated)

In [18]:
collated['segments'][0]['input_ids'].shape, collated['segments'][1]['input_ids'].shape

(torch.Size([10, 57]), torch.Size([10, 9]))

In [19]:
out.logits.shape

torch.Size([10, 66, 128256])

In [20]:
out.loss

tensor(6., dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [21]:
tokenizer.batch_decode(collated['segments'][0]['input_ids'])

['! V 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t ! |',
 '! w O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R ! |',
 '! a O : K x ! ! y A : 6 2 ! ! r O : i S ! ! W i : 1 l ! ! G J : n i ! ! p o : D D ! ! 4 3 : z k ! ! C 6 : 6 i ! |',
 '! S J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N ! |',
 '! q y : l x ! ! N b : r K ! ! 0 D : O a ! ! 7 f : V r ! ! z Z : x 7 ! ! z N : q 3 ! ! I L : n K ! ! 7 j : 3 Z ! |',
 '! q K : x l ! ! h E : l J ! ! P 9 : Q g ! ! o 6 : D G ! ! K W : 6 w ! ! L z : B W ! ! D j : x l ! ! g n : 4 o ! |',
 '! z V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r ! |',
 '! a a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R ! |',
 '! f m : Y k ! ! e J : X t ! ! 0 V : W I ! ! p d : q 3 

In [22]:
tokenizer.batch_decode(collated['segments'][1]['input_ids'])

['? ! I 9 : d t ! |',
 '? ! w M : n t ! |',
 '? ! a O : K x ! |',
 '? ! L P : 3 D ! |',
 '? ! 7 f : V r ! |',
 '? ! P 9 : Q g ! |',
 '? ! D 2 : k 7 ! |',
 '? ! Q m : W C ! |',
 '? ! 0 V : W I ! |',
 '? ! T U : f I ! |']

In [33]:
for l, m in zip(collated['segments'][1]['input_ids'], collated['segments'][1]['labels_mask']):
    print(tokenizer.decode(l[m]))


: d t ! |
: n t ! |
: K x ! |
: 3 D ! |
: V r ! |
: Q g ! |
: k 7 ! |
: W C ! |
: W I ! |
: f I ! |


In [ ]:
memory_cell = MemoryCell(config)